# 챔피언 스코어 — 클랜별 (데이터 상위 2개 클랜)

v2 하이브리드 방식을 **클랜(`guild_id`) 단위로 분리** 적용한다. 각 클랜의 데이터로
**데이터 기반 가중치·팀 실력 보정·표본 신뢰도를 클랜 내부에서 학습**하므로,
클랜별 플레이 성향이 점수에 반영된다. 데이터가 많은 **상위 N개 클랜**만 자동 선택한다.

> 방법론 상세(V1 팀 보정 / V2 퍼포먼스 / 블렌드 가중치)는 `champion_score_hybrid_v2.ipynb` 및 보고서 참조.
> 이 노트북은 그 파이프라인을 클랜별로 감싼 것이다.


# 0. 환경설정

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
pd.set_option("display.max_columns", None); pd.set_option("display.width", 180)

# 1. 파라미터 (스케줄러/재실행 시 여기만 수정)

In [2]:
PATH        = "all_participant_metric_data.csv"
TOP_N_CLANS = 2          # 데이터 많은 상위 N개 클랜만
CLAN_COL    = "guild_id" # 클랜 식별 컬럼
MIN_GAMES   = 10
W_V1, W_V2  = 0.35, 0.65
ALPHA       = 0.5
RZ_CLIP     = 4
POSITION_ORDER = ["TOP","JUG","MID","ADC","SUP"]

METRIC_DIR = {"dpm":1,"kda":1,"damage_dealt_per_death":1,"gold_per_min":1,"cs_per_min":1,
 "exp_per_min":1,"lane_gold_diff":1,"damage_to_objectives":1,"takedowns_before_15min":1,
 "dead_time_pct":-1,"vision_score":1}
METRICS = list(METRIC_DIR)

EXPERT = {
 "TOP":{"dpm":.55,"kda":.55,"damage_dealt_per_death":.75,"gold_per_min":.70,"cs_per_min":.55,"exp_per_min":.85,"lane_gold_diff":.95,"damage_to_objectives":.60,"takedowns_before_15min":.55,"dead_time_pct":.70,"vision_score":.25},
 "JUG":{"dpm":.45,"kda":.60,"damage_dealt_per_death":.60,"gold_per_min":.65,"cs_per_min":.45,"exp_per_min":.85,"lane_gold_diff":.80,"damage_to_objectives":.95,"takedowns_before_15min":.85,"dead_time_pct":.70,"vision_score":.40},
 "MID":{"dpm":.70,"kda":.55,"damage_dealt_per_death":.70,"gold_per_min":.70,"cs_per_min":.45,"exp_per_min":.85,"lane_gold_diff":.90,"damage_to_objectives":.60,"takedowns_before_15min":.65,"dead_time_pct":.70,"vision_score":.30},
 "ADC":{"dpm":.75,"kda":.55,"damage_dealt_per_death":.75,"gold_per_min":.85,"cs_per_min":.65,"exp_per_min":.85,"lane_gold_diff":.90,"damage_to_objectives":.80,"takedowns_before_15min":.50,"dead_time_pct":.70,"vision_score":.25},
 "SUP":{"dpm":.30,"kda":.60,"damage_dealt_per_death":.50,"gold_per_min":.45,"cs_per_min":.15,"exp_per_min":.80,"lane_gold_diff":.90,"damage_to_objectives":.55,"takedowns_before_15min":.55,"dead_time_pct":.70,"vision_score":.45},
}

# 2. 데이터 로드 & 상위 클랜 선택

In [3]:
df_all = pd.read_csv(PATH)
df_all = df_all[df_all["is_deleted"] == False].copy()

clan_sizes = df_all[CLAN_COL].value_counts()
TOP_CLANS = clan_sizes.head(TOP_N_CLANS).index.tolist()
print("전체 클랜 수:", df_all[CLAN_COL].nunique())
print("데이터 상위 클랜 규모:")
print(clan_sizes.head(TOP_N_CLANS).to_string())
print("\n선택된 클랜:", TOP_CLANS)

전체 클랜 수: 7
데이터 상위 클랜 규모:
guild_id
1281251734454276106    24700
936184382228693052      4070

선택된 클랜: [1281251734454276106, 936184382228693052]


# 3. 클랜별 스코어 함수

한 클랜의 데이터를 받아 라인별 챔피언 스코어를 반환한다.
모든 학습(스케일·가중치·팀보정·신뢰도)이 **그 클랜 내부**에서 이뤄진다.

In [4]:
def robust(s):
    iqr = s.quantile(0.75) - s.quantile(0.25)
    return (s - s.median()) / iqr if iqr > 0 else s * 0.0

def z_pos(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else s * 0.0

def run_for_clan(d):
    d = d.copy()
    # 방향 정렬 + 포지션 내 로버스트 스케일
    for m, sign in METRIC_DIR.items():
        if sign == -1: d[m] = -d[m]
    for m in METRICS:
        d[f"{m}_rz"] = d.groupby("position")[m].transform(robust).clip(-RZ_CLIP, RZ_CLIP)

    # 클랜 내부 데이터 가중치 → 전문가값과 블렌드
    ROLE_W = {}
    for pos in POSITION_ORDER:
        dd = d[d.position == pos]
        X = dd[[f"{m}_rz" for m in METRICS]].fillna(0).values
        y = dd.game_result.values
        if len(np.unique(y)) < 2 or len(dd) < 20:
            dw = {m: 0.0 for m in METRICS}          # 표본 부족 시 전문가값만
        else:
            lr = LogisticRegression(C=0.5, max_iter=1000).fit(X, y)
            coef = np.clip(lr.coef_[0], 0, None)
            dw = dict(zip(METRICS, coef/coef.max() if coef.max() > 0 else coef))
        ew = EXPERT[pos]; mx = max(ew.values())
        ROLE_W[pos] = {m: ALPHA*dw[m] + (1-ALPHA)*(ew[m]/mx) for m in METRICS}

    # V2 퍼포먼스
    w_arr = {pos: np.array([ROLE_W[pos][m] for m in METRICS]) for pos in POSITION_ORDER}
    rz = d[[f"{m}_rz" for m in METRICS]].values
    d["perf"] = [float(v @ w_arr[p]) for v, p in zip(rz, d.position)]

    # V1 팀 실력 보정 (클랜 내부)
    pw = d.groupby("puuid").game_result.mean(); d["p"] = d.puuid.map(pw)
    tsum = d.groupby(["custom_match_id","game_team"])["p"].transform("sum")
    tot  = d.groupby("custom_match_id")["p"].transform("sum")
    d["expected"] = (0.5 + (tsum/5 - (tot-tsum)/5)).clip(0.02, 0.98)
    d["surprise"] = d.game_result - d["expected"]

    # 집계 + 점수
    champ = d.groupby(["champion_id","position"], as_index=False).agg(
        champ_name=("champ_name","first"), games=("game_result","count"),
        unique_users=("puuid","nunique"), champion_winrate=("game_result","mean"),
        v1_core=("surprise","mean"), v2_core=("perf","mean"))
    champ = champ[champ.games >= MIN_GAMES].copy()
    if champ.empty:
        return champ, ROLE_W, None
    STABLE = int(champ.games.median())
    champ["z1"] = champ.groupby("position")["v1_core"].transform(z_pos)
    champ["z2"] = champ.groupby("position")["v2_core"].transform(z_pos)
    champ["reliability"] = np.minimum(champ.games / STABLE, 1.0)
    champ["final"] = (W_V1*champ.z1 + W_V2*champ.z2) * champ.reliability
    champ["champion_score"] = champ.groupby("position")["final"].transform(
        lambda s: 70 + z_pos(s)*15).clip(0,100).round(1)
    for c in ["champion_winrate","v1_core","v2_core"]:
        champ[c] = champ[c].round(3)
    champ = champ.sort_values(["position","champion_score"], ascending=[True,False]).reset_index(drop=True)
    return champ, ROLE_W, STABLE

# 4. 클랜별 실행

In [5]:
DISPLAY = ["champ_name","champion_id","position","games","unique_users",
           "champion_winrate","v1_core","v2_core","champion_score"]
results, weights = {}, {}
for gid in TOP_CLANS:
    champ, RW, STABLE = run_for_clan(df_all[df_all[CLAN_COL] == gid])
    champ.insert(0, CLAN_COL, gid)
    results[gid] = champ; weights[gid] = RW
    print(f"클랜 {gid}: 챔피언-라인 {len(champ)}개 | STABLE_GAMES={STABLE}")

클랜 1281251734454276106: 챔피언-라인 231개 | STABLE_GAMES=60
클랜 936184382228693052: 챔피언-라인 118개 | STABLE_GAMES=23


## 4.1 첫 번째 클랜 — 라인별 순위

In [6]:
gid = TOP_CLANS[0]
print("CLAN", gid)
results[gid][DISPLAY]

CLAN 1281251734454276106


,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,제리,CHN_166,ADC,84,27,0.571,0.069,1.816,100.0
1,자야,CHN_157,ADC,110,40,0.536,0.036,1.345,92.4
2,코그모,CHN_68,ADC,48,8,0.562,0.055,1.316,88.6
3,코르키,CHN_25,ADC,94,26,0.532,0.025,1.187,85.7
4,스웨인,CHN_130,ADC,45,14,0.622,0.117,1.003,85.6
...,...,...,...,...,...,...,...,...,...
226,문도박사,CHN_29,TOP,48,24,0.354,-0.145,0.195,50.9
227,모데카이저,CHN_82,TOP,82,37,0.402,-0.094,-0.039,47.4
228,그라가스,CHN_40,TOP,49,26,0.367,-0.126,-0.629,40.4
229,세트,CHN_119,TOP,53,17,0.321,-0.164,-0.611,35.0


## 4.2 두 번째 클랜 — 라인별 순위

In [7]:
gid = TOP_CLANS[1]
print("CLAN", gid)
results[gid][DISPLAY]

CLAN 936184382228693052


,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,카이사,CHN_56,ADC,30,14,0.700,0.182,2.830,100.0
1,징크스,CHN_55,ADC,24,14,0.583,0.098,2.176,93.3
2,유나라,CHN_171,ADC,55,19,0.527,0.038,1.867,85.7
3,트리스타나,CHN_139,ADC,18,6,0.556,0.049,1.559,80.0
4,제리,CHN_166,ADC,24,10,0.500,-0.006,1.622,79.9
...,...,...,...,...,...,...,...,...,...
113,제이스,CHN_53,TOP,11,10,0.273,-0.228,-0.156,56.8
114,쉔,CHN_121,TOP,29,13,0.448,-0.033,-0.457,50.0
115,럼블,CHN_111,TOP,27,17,0.370,-0.120,-0.164,49.2
116,암베사,CHN_13,TOP,16,11,0.188,-0.269,-0.317,47.4


## 4.3 클랜 성향 비교 (라인별 상위 3)

같은 라인이라도 클랜마다 강한 챔피언이 다르다.

In [8]:
for pos in POSITION_ORDER:
    line = [pos.ljust(4)]
    for gid in TOP_CLANS:
        t = results[gid][results[gid].position == pos].head(3)
        line.append(" / ".join(f"{r.champ_name}({r.champion_score})" for _, r in t.iterrows()))
    print(f"[{line[0]}]")
    for gid, txt in zip(TOP_CLANS, line[1:]):
        print(f"    {gid}: {txt}")

[TOP ]
    1281251734454276106: 가렌(100.0) / 갱플랭크(100.0) / 올라프(100.0)
    936184382228693052: 요네(100.0) / 카밀(100.0) / 이렐리아(87.4)
[JUG ]
    1281251734454276106: 제드(100.0) / 자헨(100.0) / 그레이브즈(97.3)
    936184382228693052: 제드(100.0) / 그레이브즈(100.0) / 제이스(97.2)
[MID ]
    1281251734454276106: 제라스(100.0) / 카사딘(100.0) / 트리스타나(92.7)
    936184382228693052: 야스오(100.0) / 제라스(91.2) / 이렐리아(87.2)
[ADC ]
    1281251734454276106: 제리(100.0) / 자야(92.4) / 코그모(88.6)
    936184382228693052: 카이사(100.0) / 징크스(93.3) / 유나라(85.7)
[SUP ]
    1281251734454276106: 이즈리얼(100.0) / 엘리스(91.9) / 멜(90.5)
    936184382228693052: 세라핀(92.8) / 제라스(92.2) / 렐(89.9)


## 4.4 클랜 성향 — 가중치 차이 (예: MID)

데이터 기반 가중치가 클랜 내부에서 학습되므로, 클랜의 성향이 다르면 가중치도 달라진다.

In [9]:
mid_cmp = pd.DataFrame({gid: weights[gid]["MID"] for gid in TOP_CLANS}).round(2)
mid_cmp.columns = [f"clan_{g}" for g in TOP_CLANS]
mid_cmp

,clan_1281251734454276106,clan_936184382228693052
dpm,0.39,0.39
kda,0.81,0.81
damage_dealt_per_death,0.39,0.39
gold_per_min,0.39,0.39
cs_per_min,0.25,0.25
exp_per_min,0.72,0.78
lane_gold_diff,0.63,0.72
damage_to_objectives,0.42,0.53
takedowns_before_15min,0.36,0.36
dead_time_pct,0.40,0.42


# 5. 저장

In [10]:
# 저장 컬럼: champ_name 으로 시작
SAVE_COLS = ["champ_name", CLAN_COL, "champion_id", "position", "games",
             "unique_users", "champion_winrate", "v1_core", "v2_core", "champion_score"]

# 클랜별 개별 저장
for gid in TOP_CLANS:
    results[gid][SAVE_COLS].to_csv(
        f"champion_score_clan_{gid}.csv", index=False, encoding="utf-8-sig")

# 통합본 (웹 업로드용) — 클랜 컬럼 포함
combined = pd.concat([results[gid][SAVE_COLS] for gid in TOP_CLANS], ignore_index=True)
combined.to_csv("champion_score_by_clan.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", [f"champion_score_clan_{g}.csv" for g in TOP_CLANS], "+ champion_score_by_clan.csv")

저장 완료: ['champion_score_clan_1281251734454276106.csv', 'champion_score_clan_936184382228693052.csv'] + champion_score_by_clan.csv


# 6. 참고

- `TOP_N_CLANS` 를 늘리면 더 많은 클랜을 자동 포함(데이터 순).
- 클랜별 표본이 작을수록 `MIN_GAMES` 를 낮추거나(예: 5) 결과 해석에 주의.
- 표본이 20행 미만인 라인은 데이터 가중치 대신 전문가값만 사용(과적합 방지).
- `champion_score` 는 **클랜 내부·라인 내부 상대 순위**다.
